<a href="https://colab.research.google.com/github/Potdooshami/1T-TaS2-point-defect-analysis/blob/master/pymaia4_day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0 - 복습
1)데이터 전처리 2)선형 회귀 3) 로지스틱 회귀

## (개발환경 세팅)

3기 데이터

In [ ]:
#지난 수업에서 썼던 코드 재사용
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import io
import pandas as pd
df = pd.read_csv(io.StringIO('''
ID,키,발 사이즈,"성별(남,여)"
차은우,167,250,여
홍지후,162,245,여
"ㅡ""ㅡ",164,255,여
  ll,168,240,여
장윤아,162,240,여
수혀니,165,230,여
미미민지,158,230,여
김노겸,159,240,여
애기공주,198,320,여
집가고싶다고,167,230,여
ㅇ,186,280,남
ㅇ[]ㅇ,161,250,여
송재민,184,290,남
정세헌,174,270,남
권하율,170,270,남
박지원,181,300,남
류민석,151,220,여
악,156,245,여
이세한,176,265,남
ㅇㅅㅇ,162,245,남
이연지,165,245,여
어니ㅏㅇ머,162,250,여
윤하준,175,265,상남자
이름,176,270,남
'''))
df.columns = ['ID','height','shoe size','sex']
gender_map = {
    '남': 'male',
    '상남자': 'male',  # '상남자'도 'male'로 매핑
    '여': 'Female'
}
df['sex'] = df['sex'].map(gender_map)
df = df.iloc[:,1:]
is_outlier = df['height'].values>190
df = df[~is_outlier]
df

4기 데이터

In [ ]:
#이번 수업에 쓸 데이터셋
import io
import pandas as pd
df_new = pd.read_csv(io.StringIO('''
kt1,169,265,m
P,158,240,f
sonny,177,270,m
jayaj,155,230,f
hi,166,245,f
erererer,165,230,f
ttttt,166,240,f
chb,158,230,f
jwjs,180,280,m
ej,157,230,f
'''), header=None, names=['name','height','shoe size','sex'])
df_new

3기, 4기 데이터 합치기

In [ ]:
df['is_male'] = (df['sex'] == 'male').astype(int)
df['is_male_t'] = df['is_male']*2-1
df['metkki'] = 3
df_new['is_male'] = (df_new['sex'] == 'm').astype(int)
df_new['is_male_t'] = df_new['is_male']*2-1
df_new['metkki'] = 4
col_common = ['height','shoe size','is_male','metkki','is_male_t']
df_common = df[col_common]
df_new_common = df_new[col_common]
df_comb = pd.concat([df_common,df_new_common])
df_comb = df_comb.reset_index(drop=True)
df_comb


## 최적의 m,c 찾기 하다가 끝남

<img src="https://github.com/MINDS-math-ai-edu/The-4th-POSTECH-Youth-Mathematical-Artificial-Intelligence-Academy-Public/blob/main/warehouse/blackboard.jpg?raw=true" width="300">

지난번 수업 때 우리가 했던 노가다

In [ ]:
m = 2# @param {type:"number"}
c = -82# @param {type:"number"}
fcn_estimate = lambda x: m*x + c
xs = df['height'].values
ys = df['shoe size'].values
def sort_by_x(x_unsorted, y_unsorted):
    """
    Sorts two lists/arrays based on the values in the first list/array.

    Args:
        x_unsorted: The list or array to sort by.
        y_unsorted: The list or array to sort according to the order of x_unsorted.

    Returns:
        A tuple containing the sorted x_unsorted and y_unsorted.
    """
    # Combine x and y into pairs, sort by x, then separate
    sorted_pairs = sorted(zip(x_unsorted, y_unsorted))
    x_sorted, y_sorted = zip(*sorted_pairs)
    return np.array(x_sorted), np.array(y_sorted)

# Example usage (optional - can be removed or commented out)
# x_unsorted = [5, 2, 8, 1, 9]
# y_unsorted = [10, 4, 16, 2, 18]
# x_sorted, y_sorted = sort_by_x(x_unsorted, y_unsorted)
# print("Sorted x:", x_sorted)
# print("Sorted y:", y_sorted)
def visualize_MSE(fcn_estimate,xs,ys,l_compare=100):
  l = l_compare
  xs,ys = sort_by_x(xs,ys)
  fcn = fcn_estimate
  fig, axs = plt.subplots(1, 2, layout='constrained',figsize=(10, 5))
  plt.sca(axs[0])
  plt.plot(xs,ys,'o',label='Actual Data')
  x_smooth = np.linspace(xs.min(), xs.max(), 100)
  plt.plot(x_smooth,fcn(x_smooth),label='Predict')
  plt.ylabel('Shoe Size')
  plt.xlabel('Height')
  xs = np.array(xs)
  ys = np.array(ys)
  errors = ys - fcn(xs)
  print(errors)
  clr_error = []
  for i, error in enumerate(errors):
      if error >= 0:
          clr = 'red'
      else:
          clr = 'blue'
      plt.plot([xs[i], xs[i]], [fcn(xs[i]), ys[i]], color=clr, linestyle='-')
      clr_error.append(clr)
  plt.legend()
  plt.title('Visualize error')





  plt.sca(axs[1])
  # 정사각형 넓이 리스트
  square_areas = errors*errors

  # 정사각형 색상 리스트
  colors = ['red', 'blue', 'green', 'yellow', 'purple']

  # 넓이 합 계산
  total_area = sum(square_areas)
  print(total_area)

  # 시각화 시작
  ax = axs[1]

  # 정사각형 그리기
  start_x = 0

  for i, area in enumerate(square_areas):
    #area = abs(errors[i])
      side = np.sqrt(area)  # 정사각형 한 변의 길이 계산
      rect = plt.Rectangle((start_x, 0), side, side, facecolor=clr_error[i])
      ax.add_patch(rect)
      start_x += side  # 다음 정사각형 시작 위치 설정

  # 텍스트 추가 (넓이 합 표시)
  #plt.text(0.5, -0.1, f'Total Area: {total_area}',
  #         horizontalalignment='center', verticalalignment='center',
  #         transform=ax.transAxes, fontsize=12)

  # 그래프 설정


  #ax.set_xlim([0, 100])
  #ax.set_ylim([0, 100])  # y축 범위 설정
  ax.set_aspect('equal')  # 정사각형 모양 유지
  plt.title('Sum of Areas of Squares')
  plt.xlabel('Width')
  plt.ylabel('Height')
  plt.plot([0, l, l, 0, 0],[0, 0, l, l, 0],'--',label=f'RMSE={l}')
  lsum = np.sqrt(square_areas.sum())
  xs_sum = np.array([0,1,1,0,0])
  ys_sum = np.array([0,0,1,1,0])
  plt.plot(xs_sum*lsum,ys_sum*lsum,label=f"RMSE(Current Model)={lsum:.3f}")
  plt.legend()

visualize_MSE(fcn_estimate,xs,ys)

In [ ]:
def calculate_rmse(xs, ys, m, c):
    """
    Vectorized RMSE calculation for linear regression models y = m*x + c.

    Args:
      xs: array-like, shape (n,)
      ys: array-like, shape (n,)
      m: array-like, shape (a,b) or scalar
      c: array-like, shape (a,b) or scalar

    Returns:
      rmse: numpy array, shape (a,b)
    """
    xs = np.asarray(xs)
    ys = np.asarray(ys)
    m = np.asarray(m)
    c = np.asarray(c)

    # xs shape: (n,), ys shape: (n,)
    # m,c shape: (a,b)
    # broadcast xs,ys to align with m,c
    # predicted y shape → (a,b,n)
    ys_pred = m[..., None] * xs + c[..., None]

    # squared error shape: (a,b,n)
    sq_err = (ys - ys_pred) ** 2

    # mean over samples (axis=-1)
    mse = np.mean(sq_err, axis=-1)
    rmse = np.sqrt(mse)
    return rmse
def cal_rmse(y_pred,y_real):
  return np.sqrt(np.mean((y_pred-y_real)**2))


def find_optimal_params_analytical(xs, ys):
  """
  Finds the optimal m and c for a linear regression model (y = mx + c)
  using the analytical solution (Normal Equation).

  Args:
    xs: A numpy array or list of x-values.
    ys: A numpy array or list of y-values.

  Returns:
    A tuple containing the optimal m and c values.
  """
  xs = np.array(xs)
  ys = np.array(ys)

  # Add a column of ones to xs for the intercept term (c)
  X = np.vstack([xs, np.ones(len(xs))]).T

  # Calculate the parameters using the normal equation: (X^T * X)^(-1) * X^T * y
  XTX = X.T @ X
  XTy = X.T @ ys
  params = np.linalg.inv(XTX) @ XTy

  m = params[0]
  c = params[1]

  return m, c

# Example usage (optional - can be removed or commented out)
# xs_example = np.array([1, 2, 3, 4, 5])
# ys_example = np.array([2, 4, 5, 4, 5])
# optimal_m_analytical, optimal_c_analytical = find_optimal_params_analytical(xs_example, ys_example)
# print(f"Optimal m (Analytical): {optimal_m_analytical:.3f}, Optimal c (Analytical): {optimal_c_analytical:.3f}")

def annotate_mco(m_val,c_val,obj_val):
  txt = plt.annotate(f'({m_val:.3f},{c_val:.3f});{obj_val:.3f}', (m_val, c_val), textcoords="offset points", xytext=(5,5), ha='left')

def plot_trajectory(m_values,c_values,obj_values):

  # Extract m and c values


  # Create a scatter plot
  plt.figure(figsize=(8, 6))
  plt.scatter(m_values, c_values, marker='o')

  # Add labels and title
  plt.xlabel('m')
  plt.ylabel('c')
  plt.title('Trajectory of (m, c) Parameters')

  # Add annotations for each point with step index and coordinates
  for i, (m_val, c_val,obj_val) in enumerate(zip(m_values, c_values,obj_values)):
      txt = plt.annotate(f'{i}:({m_val:.3f},{c_val:.3f});{obj_val:.3f}', (m_val, c_val), textcoords="offset points", xytext=(5,5), ha='left')
      print(obj_val)


  # Show the plot
  plt.grid(True)

In [ ]:
mcList = np.array([(2,-80),(2,-82),(2,-81.5),(2,-81.7),(1.965,-76),(1.96568,-76.16)])
m_values = mcList[:,0]
c_values = mcList[:,1]
obj_values = calculate_rmse(xs, ys, m_values, c_values)

plot_trajectory(m_values,c_values,obj_values)
fig = plt.gcf()
ax = plt.gca()


# 1 - 모델 최적화

정답 공개

In [ ]:
m,c = find_optimal_params_analytical(xs, ys)
obj = calculate_rmse(xs, ys, m, c)
plt.sca(ax)
ax.scatter(m,c,c='r')
annotate_mco(m,c,obj)
fig

## 정확해 구하기

$
a = \frac{n \sum x_i y_i - \sum x_i \sum y_i}{n \sum x_i^2 - (\sum x_i)^2}
$

$
b = \frac{\sum y_i - a \sum x_i}{n}
$

$ a = \dfrac{\mathrm{Cov}(x, y)}{\mathrm{Var}(x)} $

$ b = \bar{y} - a \bar{x} $


[증명](https://github.com/MINDS-math-ai-edu/The-4th-POSTECH-Youth-Mathematical-Artificial-Intelligence-Academy-Public/blob/main/warehouse/svm3d.png)

## 경사하강법(반복적 최적화 대표)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 손실 함수 (y = x^2)
def loss(x):
    return 1*x**2

# 손실 함수의 기울기
def gradient(x):
    return 1**2 * x

# 경사하강법
x = 3  # 초기값
learning_rate = 1# @param {type:"raw"}
steps = 10
history = [x]

for _ in range(steps):
    grad = gradient(x)
    x -= learning_rate * grad  # 업데이트
    history.append(x)

# 시각화
x_vals = np.linspace(-3, 3, 100)
y_vals = loss(x_vals)

# 컬러맵 데이터 준비
steps_array = np.arange(len(history))  # 단계 수를 나타내는 배열
colors = plt.cm.viridis(steps_array / len(history))  # 단계별 색상 매핑

plt.plot(x_vals, y_vals, label="Loss Function")
plt.plot(history,[loss(x) for x in history])
scatter = plt.scatter(history, [loss(x) for x in history], c=steps_array, cmap="viridis", s=50, zorder=5,label='Step')

# 컬러바 추가
cbar = plt.colorbar(scatter)
cbar.set_label("Step Number")

plt.title("Gradient Descent Visualization with Color Gradient")
plt.xlabel("Parameter (x)")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 과적합

In [ ]:
import seaborn as sns
ax = sns.scatterplot(data=df_comb, x='height', y='shoe size',style='metkki',hue='is_male')
plt.show()


In [ ]:
plt.sca(ax)

# Assuming xs and ys are already defined from your previous cells
# If not, you would need to load or define them here.
# Example:
xs = df['height'].values
ys = df['shoe size'].values

# Fit a 10th-degree polynomial to the data
degree = 10
coefficients = np.polyfit(xs, ys, degree)
poly_model = np.poly1d(coefficients)

# Generate points for the fitted curve
x_curve = np.linspace(xs.min(), xs.max(), 100)
y_curve = poly_model(x_curve)

# Plot the original data and the fitted curve
#plt.figure(figsize=(10, 6))
#plt.scatter(xs, ys, label='Original Data')
plt.plot(x_curve, y_curve, color='red', label=f'{degree}th Degree Polynomial Fit')
m_anal,c_anal = find_optimal_params_analytical(xs, ys)
plt.plot(x_curve,m_anal*x_curve+c_anal,color='green',label='Analy/tical')
plt.xlabel('Height')
plt.ylabel('Shoe Size')
plt.title(f'Polynomial Regression (Degree {degree}) - Demonstrating Overfitting')
plt.legend()
plt.grid(True)
plt.show()


과적합 여부 판단하기

In [ ]:
dfs = [df,df_new]
dfs_label =['3기 데이터 예측하기','4기 데이터 예측하기']
estimators = [poly_model,lambda x: m_anal*x + c_anal]
estimators_label = ['10차 다항식 모델','선형 모델']
for imodel,(df_now,dflabel) in enumerate(zip(dfs,dfs_label)):
  for ipred,(estimator,est_label) in enumerate(zip(estimators,estimators_label)):
    xs = df_now['height'].values
    ys = df_now['shoe size'].values
    rmse = cal_rmse(estimator(xs),ys)
    print(f"'{est_label}' 로 '{dflabel}' - > Loss = {rmse:2f}" )


training - test 나누기 필요

# 2 - SVM 실습

## 마진 최대화

In [ ]:
df_fcs = df_new

m = 1 #@param {type:"number"}
c = -250 #@param {type:"number"}
is_croped_view = True #@param {type:"boolean"}
vis_SVM = True
vis_softmax = False
def visualize_SVM(df_fcs,m,c,is_croped_view,vis_SVM,vis_softmax):
  ax = sns.scatterplot(data=df_fcs, x='shoe size', y='is_male_t',alpha=.3,s=100,label='training data')
  ax.axhline(0,c='k')
  ax.axhline(1,c='k')
  ax.axhline(-1,c='k')

  walls =  [(1-c)/m,(-1-c)/m]
  ts = df_fcs['is_male_t'].values
  xs = df_fcs['shoe size'].values
  if (ts*(m*xs+c)>=1).all():
    c_wall = 'g'
  else:
    c_wall = 'r'

  x_curve = np.linspace(df_fcs['shoe size'].min(), df_fcs['shoe size'].max(), 100)
  y_curve = m*x_curve+c
  y_curve_logistic = 1/(1+np.exp(-(m*x_curve+c)))
  y_curve_logistic_t = y_curve_logistic*2 -1
  if vis_SVM:
    plt.plot(x_curve,y_curve,color=c_wall,label=f's = {m:.3f}x+{c:.3f}')
    plt.axvline(-c/m,color=c_wall,linestyle=':',label='hyper plane')
    plt.axvspan(walls[0],walls[1],alpha=.2,color=c_wall,label='margin')
  if vis_softmax:
    plt.plot(x_curve,y_curve_logistic_t,color='b',linewidth=3,label='sigmoid(s)')
  if is_croped_view:
    plt.ylim(-2,2)
    plt.xlim(220,290)
  plt.title('predict shoe size by sex using 3gi data')
  plt.legend()
visualize_SVM(df_new,m,c,is_croped_view,vis_SVM,vis_softmax)

로지스틱 회귀 vs SVM

In [ ]:
df_fcs = df_new

m = 1 #@param {type:"number"}
c = -250 #@param {type:"number"}
is_croped_view = True #@param {type:"boolean"}
vis_SVM = True #@param {type:"boolean"}
vis_softmax = True #@param {type:"boolean"}
visualize_SVM(df_new,m,c,is_croped_view,vis_SVM,vis_softmax)

## hard margin의 한계

In [ ]:

df_fcs = df_comb[df_comb['metkki']==3]
m = 1 #@param {type:"number"}
c = -250 #@param {type:"number"}
is_croped_view = True #@param {type:"boolean"}
vis_SVM = True #@param {type:"boolean"}
vis_softmax = True #@param {type:"boolean"}
visualize_SVM(df_fcs,m,c,is_croped_view,vis_SVM,vis_softmax)
plt.title('predict 4gi')

## soft margin SVM 과 하이퍼파라미터 C

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
scalers = {
    'null' : FunctionTransformer(),
    'std' : StandardScaler()
}
predictors ={
    'softmax' : LogisticRegression( solver='lbfgs'),
    'svm' : LinearSVC(C=1.0, random_state=42),
    'svm_hard' : LinearSVC(C=1000000.0, random_state=42,max_iter=5000),
    'svm_soft' : LinearSVC(C=0.001, random_state=42)
}


In [ ]:
def get_original_coefs(scaler, linear_model):
    """
    StandardScaler로 학습된 선형 모델의 계수(W, b)를
    원본 데이터 스케일로 역변환합니다.

    Args:
        scaler (StandardScaler): fit이 완료된 스케일러 객체
        linear_model (object): .coef_와 .intercept_를 가진,
                               fit이 완료된 선형 모델 객체
                               (e.g., LinearSVC, LogisticRegression)

    Returns:
        tuple: (w_original, b_original)
    """

    # 1. 스케일러에서 평균(mu)과 표준편차(sigma) 추출
    mu = scaler.mean_
    sigma = scaler.scale_

    # 2. 모델에서 스케일링된 계수(w')와 절편(b') 추출
    #    .coef_가 (1, D) 형태일 수 있으므로 .ravel()로 1D 배열로 만듭니다.
    w_prime = linear_model.coef_.ravel()
    b_prime = linear_model.intercept_

    # 3. 원본 계수(W, b)로 변환
    #    (W = W' / sigma)
    w_original = w_prime / sigma

    #    (b = b' - sum(W' * mu / sigma) = b' - sum(W_original * mu))
    b_original = b_prime - np.sum(w_original * mu)

    # 사용자의 1D 문제에 맞게 (1, D) 형태로 다시 맞춰주거나,
    # 1D 벡터 그대로 반환할 수 있습니다.
    # 여기서는 원본 모델의 .coef_ 형태와 맞춰줍니다.
    return w_original.reshape(linear_model.coef_.shape), b_original

In [ ]:
df_fcs = df_comb[df_comb['metkki']==3]
X = df_fcs['shoe size'].values.reshape(-1,1)
y = df_fcs['is_male'].values
#X = scalers['null'].fit_transform(X)
scaler = scalers['std']
X = scaler.fit_transform(X)
predicted = dict()
for ind,(key,predictor) in enumerate(predictors.items()):
  predictor = predictors[key]
  predictor.fit(X,y)
  predicted[key] =dict()
  predicted[key]['predictor'] = predictor

  m_o,c_o = get_original_coefs(scaler, predictor)
  predicted[key]['m_origin'] = m_o
  predicted[key]['c_origin'] = c_o



In [ ]:
key = 'svm'#@param ['softmax','svm_soft','svm','svm_hard']
m = predicted[key]['m_origin'][0,0]
c = predicted[key]['c_origin'][0]
is_croped_view = True #@param {type:"boolean"}
vis_SVM = True #@param {type:"boolean"}
vis_softmax = True #@param {type:"boolean"}
visualize_SVM(df_fcs,m,c,is_croped_view,vis_SVM,vis_softmax)
plt.title('predict 4gi')

# 3 - 더 복잡한 모델

## 다변수 분류

키,발 사이즈 모두를 사용해서 성별 예측하기

In [ ]:
def visualize_SVM_wall2d(w1,w2,b,xLim,c='k',**args):
  ls =[':','-',':']
  for indHP in [0, 1, 2]:
    x1_hp = np.array(xLim)
    x2_hp = (-w1 * x1_hp - b+indHP-1) / w2
    plt.plot(x1_hp,x2_hp,linestyle = ls[indHP],c=c,**args)

In [ ]:
predictor

In [ ]:
import seaborn as sns
df_fcs = df_comb[df_comb['metkki']==3]
ax = sns.scatterplot(data=df_fcs, x='height', y='shoe size',style='metkki',hue='is_male')

X = df_fcs[['height','shoe size']].values
y = df_fcs['is_male'].values
scaler = scalers['std']
X = scaler.fit_transform(X)
predictor = LinearSVC(C=1, random_state=42)
predictor = predictors['softmax']
predictor.fit(X,y)

w,b = get_original_coefs(scaler, predictor)
#w = predictor.coef_
#b = predictor.intercept_[0]
w1 = w[0,0]
w2 = w[0,1]

visualize_SVM_wall2d(w1,w2,b,xLim=(150,190),c='k')


변수 스케일링을 해야함

## 다중 클래스 분류

시각화를 포기

![](https://sundeeppothula1993.github.io/ARTML//assets/img/iris.png)

In [ ]:
import pandas as pd
from sklearn import datasets

# Iris 데이터셋 로드
iris = datasets.load_iris()

# DataFrame으로 변환
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
#df['target_name'] = df.target.map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

# 데이터 확인
df.columns
className = ['setosa', 'versicolor', 'virginica']
df

xColinds = [0,3]
xminmax = np.array([df.iloc[:,xColinds].min(),df.iloc[:,xColinds].max()])
df.iloc[:,xColinds]
print(df.iloc[:,-1])
lbl = df.columns[xColinds]
print(lbl)

$
w_1x_1+w_2x_2 + b=0
$

$
w_1x_1+w_2x_2 + b=1
$

$
w_1x_1+w_2x_2 + b=-1
$


In [ ]:
xc = xColinds;
w11 = 1 # @param {type:"raw"}
w21 = -2. # @param {type:"raw"}
b1 = 1 # @param {type:"raw"}

w12 = -1 # @param {type:"raw"}
w22 = 1 # @param {type:"raw"}
b2 = 2.5 # @param {type:"raw"}

w13 = -1.02959295 # @param {type:"raw"}
w23 = 2.91132701 # @param {type:"raw"}
b3 = -1.7983700648051586 # @param {type:"raw"}



wb = np.array([[w11, w21, b1],
               [w12, w22, b2],
               [w13, w23, b3]])



x1 = df.iloc[:,xc[0]]
x2 = df.iloc[:,xc[1]]
c1 = df.iloc[:,-1]
c2 = df.iloc[:,-1] == 0
clrs = ['red','green','blue']
ls = ['--','-','--']

fig,axs = plt.subplots(1,3,figsize=(15,4))



for fcsClass in range(3):
  plt.sca(axs[fcsClass])
  for indClass in range(3):
    if fcsClass == indClass:
      mk = 'o'
    else:
      mk = 'x'
    posClass = df.iloc[:,-1] == indClass
    x1_ = x1[posClass]
    x2_ = x2[posClass]
    plt.scatter(x1_,x2_,c=clrs[indClass],marker = mk)
  w1 = wb[fcsClass,0]
  w2 = wb[fcsClass,1]
  b = wb[fcsClass,2]
  print(w1,w2,b)
  x1_hp = xminmax[:,0]
  for indHP in [0, 1, 2]:
    x2_hp = (-w1 * x1_hp - b+indHP-1) / w2
    plt.plot(x1_hp,x2_hp,c=clrs[fcsClass],linestyle = ls[indHP])
  #plt.xlim([4, 8.2])  # xlim 설정
  #plt.ylim([0, 2.6])  # ylim 설정
  ax = plt.gca()
  ax.set_aspect('equal', adjustable='box')
  if fcsClass == 1:
    plt.legend(className)
  plt.xlabel(lbl[0])
  plt.ylabel(lbl[1])
  #plt.xlim([4, 8.2])  # xlim 설정
  #plt.ylim([0, 2.6])  # ylim 설정





  #ax.set_aspect('equal')


## 마음 속에 시각화 하기

[변수가 3개?](https://github.com/MINDS-math-ai-edu/The-4th-POSTECH-Youth-Mathematical-Artificial-Intelligence-Academy-Public/blob/main/warehouse/svm3d.png)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
indices_list = ([0,1],[0,1,2,3])

for ind,indices in enumerate(indices_list):
  X_train, X_test, y_train, y_test = train_test_split(
      df.iloc[:,indices], df.iloc[:,-1], test_size=0.25, random_state=0
  )
  scaler = scalers['std']
  X_train = scaler.fit_transform(X_train)
  X_test = scaler.transform(X_test)
  predictor = predictors['svm']
  predictor.fit(X_train,y_train)

  print(f'{df.columns[indices]}를 이용해서 학습시킨 정확도',predictor.score(X_test,y_test),' ')